In [0]:
%sql
DROP TABLE IF EXISTS capstone.silver.transactions;
CREATE TABLE capstone.silver.transactions
USING DELTA
AS
WITH raw AS (
  SELECT
    order_id,
    item_id,
    quantity,
    price,
    trim(order_timestamp) order_timestamp,
    corrupted_flag,
    _ingest_timestamp,
    _source_file_name
  FROM capstone.bronze.transactions
),
tz_mapped AS (
  SELECT
    order_id,
    item_id,
    quantity,
    price,
    -- Replace common zone names with offsets so to_timestamp(..., 'yyyy-MM-dd HH:mm:ssXXX') can parse them
    regexp_replace(
      regexp_replace(
        regexp_replace(
          regexp_replace(order_timestamp, 'PST', '-08:00'),
        'PDT', '-07:00'),
      'EST', '-05:00'),
    'EDT', '-04:00') AS order_timestamp_offset,
    corrupted_flag,
    _ingest_timestamp,
    _source_file_name
  FROM raw
),
cleaned AS (
  SELECT
    order_id,
    item_id,
    TRY_CAST(regexp_replace(quantity, '[^0-9]', '') AS INT) AS quantity_int,
    TRY_CAST(regexp_replace(price, '[$,]', '') AS DECIMAL(10,2)) AS price_dec,
    to_utc_timestamp(try_to_timestamp(order_timestamp_offset, 'yyyy-MM-dd HH:mm:ss XXX'), 'UTC') AS order_timestamp_utc,
    corrupted_flag,
    _ingest_timestamp,
    _source_file_name
  FROM tz_mapped
)
SELECT
  order_id,
  item_id,
  quantity_int AS quantity,
  price_dec AS price,
  order_timestamp_utc AS order_timestamp,
  _ingest_timestamp,
  _source_file_name
FROM cleaned
WHERE corrupted_flag IS NULL OR corrupted_flag <> 'Y';

In [0]:
print("Silver Table Counts:")
print("capstone.silver.transactions: ", spark.table("capstone.silver.transactions").count())